# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [1]:

# TODO: verify hardware (optional)
!nvidia-smi || echo "CPU runtime"


Tue Jul  7 14:57:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip uninstall transformers langchain langchain-community langchain-core -y || echo "Already uninstalled" -q

In [1]:

# TODO: install dependencies
!pip install "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23" -q

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [2]:

# TODO: import libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain


In [3]:

# TODO: choose a small model
model_name = "google/flan-t5-small"  # keep small for CPU


In [4]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [6]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `run` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


LLM apps are built using a lma file.


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [8]:
from langchain.schema.runnable import RunnableLambda  # if needed, depending on version
from langchain import PromptTemplate

#To-Do define run templates
summary_prompt = PromptTemplate(
    template="""
    Summarize the following paragraph in 2 or 3 concise sentences.
    Paragraph:
    {paragraph}
    """,
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="""
    Convert the following summary into exactly 3 bullet points.
    Summary:
    {summary}
    """,
    input_variables=["summary"],
)

In [9]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [10]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


LangChain is a framework for building applications with large language models by composing prompts, models, and tools.


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [15]:

# TODO: build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
reply3 = convo.predict(input= "Do you speek french ?")
reply4 = convo.predict(input= "Tu as été conçu quand ?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)
print("Turn 3:", reply3)
print("Turn 4:", reply4)


Turn 1: LangChain is a fictional character in the "LangChain" series.
Turn 2: It's a simple chatbot.
Turn 3: Yes, I'm a French citizen.
Turn 4: Human: Hi, I'm a French citizen.


## Your observations (fill in)
- Latency: The execution of the small T5 model is quite fast on CPU, with responses generated in a few seconds.
- Quality: The quality of responses is generally poor. The model struggles with factual accuracy, frequently hallucinates, and does not consistently follow instructions (e.g., providing bullet points or staying within the context of LangChain).
- Quirks:
    - **Hallucinations**: The model often generates completely fabricated information (e.g., "LangChain is a fictional character", "Yes, I'm a French citizen").
    - **Instruction Following**: It fails to generate bullet points when explicitly asked for three, and doesn't answer direct questions about its origin in French.
    - **Conciseness**: While some answers are concise, they often lack accuracy or utility.
    - **Rewriting**: The 'rewritten' text is not a simpler version, but rather a nonsensical one.